# Reproducing SupplyMind Benchmarks

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShAuRyA-Noodle/Sleep-Token/blob/main/notebooks/03_reproducing_benchmarks.ipynb)

This notebook reproduces every benchmark number from the paper with exact seeds.

In [1]:
# [nb-exec-patched]
# Colab installs deps here; a local venv already has them. Guarded so a local run
# never downgrades a shared environment (e.g. pinned trl) or requires network.
try:
    import google.colab  # noqa: F401
    _IN_COLAB = True
except ImportError:
    _IN_COLAB = False
if _IN_COLAB:
    import subprocess, sys as _sys
    subprocess.run([_sys.executable, '-m', 'pip', 'install', '-q', 'gymnasium==0.29.1', 'stable-baselines3==2.2.1', 'sb3-contrib==2.2.1', 'torch', 'scipy', 'networkx', 'numpy', 'pydantic', 'fastapi'], check=False)
    print('install ok (Colab)')
else:
    print('Local venv detected — skipping pip install (deps already present).')


Local venv detected — skipping pip install (deps already present).


In [ ]:
import numpy as np
import sys, os
# [nb-exec-patched] The env loads server/data/graphs/*.json via cwd-relative paths, so detect the
# repo root and chdir to it. This makes the notebook launcher-independent: a standalone
# `jupyter nbconvert --execute notebooks/03_...ipynb` (kernel cwd = notebooks/) now works.
_here = os.getcwd()
_root = _here if os.path.exists(os.path.join(_here, 'server')) else os.path.dirname(_here)
os.chdir(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
print(f'Repo root: {_root}; cwd={os.getcwd()}')

from server.supply_environment import SupplyMindEnvironment
from scripted_agent import choose_action

TASKS = ['easy_typhoon_response', 'medium_multi_front', 'hard_cascading_crisis']
# [nb-exec-patched] smoke: reduced seeds/episodes so the medium+hard scripted grid finishes
# in minutes on CPU (hard ~6s/episode). Full reproduction: SEEDS=[42,99,7,123,256], N_EPISODES=20.
SEEDS = [42, 99, 7]
N_EPISODES = 5

In [3]:
# Benchmark: Scripted Agent
env = SupplyMindEnvironment()
results = {}

for task_id in TASKS:
    scores = []
    for seed in SEEDS:
        for ep in range(N_EPISODES):
            obs = env.reset(task_id=task_id, seed=seed * 1000 + ep)
            step = 0
            while not obs.done:
                action = choose_action(obs, step)
                obs = env.step(action)
                step += 1
            grade = env.grade()
            scores.append(grade['score'])
    results[task_id] = scores
    print(f'{task_id}: {np.mean(scores):.4f} +/- {np.std(scores):.4f} (n={len(scores)})')

easy_typhoon_response: 0.6819 +/- 0.0708 (n=100)


medium_multi_front: 0.6300 +/- 0.0886 (n=100)


KeyboardInterrupt: 

In [ ]:
# Statistical significance
from scipy.stats import wilcoxon

scripted_all = []
for scores in results.values():
    scripted_all.extend(scores)

# Bootstrap 95% CI
rng = np.random.default_rng(42)
boot_means = [np.mean(rng.choice(scripted_all, len(scripted_all))) for _ in range(1000)]
ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5])
print(f'\nScripted Agent Overall: {np.mean(scripted_all):.4f}')
print(f'95% Bootstrap CI: [{ci_low:.4f}, {ci_high:.4f}]')

In [ ]:
# Backtesting: Calibration error
from benchmark.backtesting import simulate_crisis, compute_calibration_error, HISTORICAL_CRISES

for crisis_id, crisis in HISTORICAL_CRISES.items():
    sim = simulate_crisis(crisis_id, n_runs=20)
    cal = compute_calibration_error(sim, crisis['ground_truth'])
    print(f'{crisis["name"]}: {cal["mean_relative_error_pct"]:.1f}% calibration error')